# Data preparation, retrieval, and OpenAI answer generation

Follow the full pipeline: local catalog ? validated documents ? index mapping ? optional ingestion and embeddings ? retrieval ? model context ? checked OpenAI answer. Use the project .venv kernel. Credentials stay in .env. Index creation and ingestion are disabled by default because your products are already indexed. Live searches and generation repeat API calls when rerun.


## 1. Set up the notebook
Locate the project from either the repository root or the notebooks folder, then import
its existing functions. No search or generation happens in this group.


In [ ]:
import json
import sys
from pathlib import Path

from IPython.display import display

working_directory = Path.cwd().resolve()
project_root = next(
    (folder for folder in [working_directory, *working_directory.parents]
     if (folder / "src" / "config.py").is_file()
     and (folder / "pyproject.toml").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Open this notebook from the elastic-search project folder.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.answer import generate_answer
from src.catalog import load_catalog, index_document
from src.config import load_settings
from src.context import build_context
from src.diagnostics import error_message, inspect_project, index_definition
from src.elastic_client import create_client
from src.search import Constraints, build_request, run_search
from scripts.create_index import main as create_index_main
from scripts.ingest import main as ingest_main

print("Project:", project_root)
print("Kernel Python:", sys.executable)


## 2. Load and validate the local catalog

The Product, Variant, and Attributes models in src/catalog.py define valid product data. load_catalog() reads JSONL, validates records, and rejects duplicate IDs, duplicate sizes, or an empty catalog. Inspect one product and its size-specific stock below. Nothing is uploaded.


In [ ]:
products = load_catalog(project_root / "data" / "products.jsonl")
print("Validated products:", len(products))
sample_product = products[0]
display(sample_product.model_dump())


## 3. Prepare searchable documents

index_document() combines each product name, description, and guide into searchable_content. The resulting dictionaries are what ingestion uploads. This does not modify products.jsonl or generate vectors locally.


In [ ]:
documents = [index_document(product) for product in products]
print("Prepared documents:", len(documents))
display(documents[0])
print("Searchable text:\n", documents[0]["searchable_content"])


## 4. Load settings and prepare the index mapping

Load .env without displaying keys, and read config/products-index.json through index_definition(). It replaces the endpoint placeholder. searchable_content is the field name; semantic_text is its Elastic type. The configured Jina endpoint generates embeddings during indexing. Price is numeric and variants are nested. This cell prepares configuration without changing Elastic.


In [ ]:
settings = load_settings()
definition = index_definition(settings)
print("Target index:", settings.es_index)
print("Answer model:", settings.openai_model)
display(definition)


## 5. Optional index creation and ingestion

Leave both flags False for your existing index. For a fresh index choose a new ES_INDEX in .env, restart the kernel to clear stale environment values, and enable both flags. These cells call the existing scripts: creation refuses an existing index; ingestion validates and prepares products again and uploads batches of ten. create operations refuse duplicate IDs with HTTP 409. Elastic handles chunking and Jina embeddings for semantic_text, then refresh makes documents searchable. Partial successes remain indexed on failure. Enabling these flags makes live writes; there is no deletion or overwrite.


In [ ]:
create_new_index = False
upload_catalog = False
try:
    if create_new_index:
        create_index_main()
    else:
        print("Index creation skipped.")
    if upload_catalog:
        ingest_main()
    else:
        print("Ingestion skipped; using indexed products.")
except Exception as exc:
    raise RuntimeError(error_message(exc)) from None


## 6. Verify the indexed deployment

Inspect the actual endpoint, mapping, settings, and document count. Local preparation and indexed data are separate: changing the source catalog does not update search until ingestion. This does not yet verify OpenAI generation access.


In [ ]:
try:
    with create_client(settings) as client:
        deployment = inspect_project(client, settings)
    if not deployment["index_exists"]:
        raise ValueError("The configured index does not exist. Check ES_URL and ES_INDEX.")
except Exception as exc:
    raise RuntimeError(error_message(exc)) from None

print("Index:", settings.es_index)
print("Products indexed:", deployment["product_count"])
print("Answer model:", settings.openai_model)
display(deployment)


## 7. Choose the question and retrieval constraints
Edit the values below. `mode` can be `semantic`, `keyword`, or `hybrid`.
Use `None` for no budget or size restriction. Filters come from these values, not automatic
extraction from the question. Keep the wording and controls consistent.
The second cell previews the request without sending it.


In [ ]:
question = "Comfortable shoes for walking around New York all day"
mode = "hybrid"
limit = 5
constraints = Constraints(
    max_price=100,
    currency="USD",
    size="9",
    size_system="US",
    in_stock=True,
)


In [ ]:
request = build_request(question, mode, constraints, limit)
print(json.dumps(request, indent=2, ensure_ascii=False))


## 8. Retrieve and inspect evidence
Search the existing index using the selected mode and filters. Semantic retrieval uses
Elasticsearch's configured embedding endpoint; hybrid combines semantic and keyword ranks.
Inspect passages and inventory before generating an answer. Scores are ranking values,
not confidence probabilities, and are not directly comparable across modes.


In [ ]:
# Clear downstream values so a failed rerun cannot leave an old answer ready to display.
result = None
context = None
generated = None
try:
    with create_client(settings) as client:
        result = run_search(client, settings.es_index, question, mode, constraints, limit)
except Exception as exc:
    raise RuntimeError(error_message(exc)) from None

print("Submitted question:", result["question"])
print("Mode:", result["mode"], "| Filters:", result["constraints"])
print("Elasticsearch time (ms):", result["es_took_ms"])
print("Client elapsed (ms):", result["elapsed_ms"])
if not result["products"]:
    print("No products matched these constraints.")
for product in result["products"]:
    print("\n" + product["name"], "|", product["product_id"])
    print(f"{product['currency']} {product['price']:.2f} | Score: {product['score']}")
    for passage in product["passages"] or [product["description"]]:
        print(passage)
    display({"variants": product["variants"], "attributes": product["attributes"]})


## 9. Build the model context
Keep product facts, matching size/stock variants, and evidence passages. Preview exactly
what the answering model receives, alongside the instructions from `prompts/answer.txt`.
Availability is indexed demo data, not a reservation.


In [ ]:
if result is None:
    raise RuntimeError("Run retrieval successfully before building context.")
context = build_context(result)
print(json.dumps(context, indent=2, ensure_ascii=False))


In [ ]:
print((project_root / "prompts" / "answer.txt").read_text(encoding="utf-8"))


## 10. Generate and validate the answer
Make a real OpenAI call using the prepared evidence and existing structured-answer schema.
`generate_answer()` checks completion, product citations, duplicate recommendations,
and exact evidence quotes. It also handles empty evidence without inventing products.
These checks do not prove every generated claim; review the explanation yourself.


In [ ]:
generated = None
if context is None:
    raise RuntimeError("Build the context before generating an answer.")
try:
    generated = generate_answer(settings, context)
except Exception as exc:
    raise RuntimeError(error_message(exc)) from None
print("Answer generated and citation checks passed.")


## 11. Read the final answer and usage
Display the explanation and recommendations. Names, prices, and eligible inventory come
from the retrieved context. Also inspect missing information, generation timing, and tokens.
Notebook outputs may be saved with the file; clear outputs before sharing if needed.


In [ ]:
if generated is None:
    raise RuntimeError("Generate an answer successfully before displaying it.")
answer = generated["answer"]
print(answer["explanation"])
sources = {product["product_id"]: product for product in context["products"]}
for recommendation in answer["recommendations"]:
    product = sources[recommendation["product_id"]]
    print(f"\n{product['name']} | {product['currency']} {product['price']:.2f}")
    print("Reason:", recommendation["reason"])
    print(f"Evidence [{product['product_id']}]: {recommendation['evidence_quote']}")
    print("Eligible inventory:", product["eligible_variants"])
if answer["missing_information"]:
    print("\nMissing information:")
    for item in answer["missing_information"]:
        print("-", item)
print("\nModel:", generated["model"])
print("Generation elapsed (ms):", generated["elapsed_ms"])
display(generated["usage"])
